In [ ]:
pip install pypdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/code_penal_togo.pdf")

print(len(reader.pages))

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/code_penal_togo.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

In [ ]:
with open("data/code_penal.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
# Rechargement
with open("data/code_penal.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
pattern = r'(Article\s+(?:\d+(?:er)?|premier))'

In [ ]:
import re

articles = re.split(pattern, text)

In [ ]:
article_chunks = []

for i in range(1, len(articles), 2):
    article_title = articles[i]
    article_content = articles[i+1]

    article_chunks.append(article_title + article_content)

In [ ]:
# vérification
print(len(article_chunks))
print(article_chunks[0])

In [ ]:
import json

with open("data/articles.json", "w", encoding="utf-8") as f:
    json.dump(article_chunks, f, ensure_ascii=False, indent=2)

In [ ]:
pip install sentence-transformers faiss-cpu

In [ ]:
import json

with open("data/articles.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

print(len(articles))
print(articles[0])

In [ ]:
pip install pandas

# Utilisation d'une API pour l'embeddings au lieu de SentenceTransformer


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import numpy as np

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

embeddings = []

for article in articles:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=article
    )

    embeddings.append(
        response.data[0].embedding
    )

embeddings = np.array(
    embeddings,
    dtype="float32"
)

print(embeddings.shape)

In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

In [ ]:
# Sauvegarde de l'index FAISS
import os

os.makedirs("vector_db", exist_ok=True)

faiss.write_index(
    index,
    "vector_db/code_penal.index"
)

print("Index sauvegardé")

In [ ]:
query = "injure publique"

query_embedding = model.encode([query])

D, I = index.search(query_embedding, k=5)

for i in I[0]:
    print(articles[i])
    print("\n-----------------\n")

In [ ]:
def retrieve_articles(query, k=5):

    query_embedding = model.encode([query])

    D, I = index.search(query_embedding, k)

    results = [articles[i] for i in I[0]]

    return results

In [ ]:
retrieve_articles("insulte publique")

In [ ]:
def build_context(docs):

    context = "\n\n".join(docs)

    return context

In [ ]:
def build_prompt(question, context):

    prompt = f"""
Tu es un assistant juridique spécialisé dans le droit togolais.

Réponds uniquement à partir des articles fournis.

Articles :
{context}

Question :
{question}

Réponse :
"""
    return prompt

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3",
        "prompt": "Explique le vol en droit pénal",
        "stream": False
    }
)

print(response.json()["response"])

In [ ]:
def rag_pipeline(question):

    query_embedding = model.encode([question])

    D, I = index.search(query_embedding, k=3)

    docs = [articles[i] for i in I[0]]

    context = "\n\n".join(docs)

    prompt = f"""
Tu es un assistant juridique spécialisé dans le droit togolais.

Réponds uniquement à partir des articles fournis.
Si l'information n'est pas dans les articles, dis que tu ne sais pas.

Cite les numéros d'articles lorsque c'est possible.

Articles :
{context}

Question :
{question}

Réponse :
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )

    answer = response.json()["response"]

    return answer, docs

In [ ]:
answer, sources = rag_pipeline(
    "Si quelqu'un insulte une personne en public que risque-t-il ?"
)

print(answer)

print("\nArticles utilisés :\n")

for s in sources:
    print(s)
    print("\n----------------\n")

In [ ]:
import openai
from dotenv import load_dotenv

print("OpenAI OK")
print("dotenv OK")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENAI_API_KEY")[:15])

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Connexion OK")

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "Bonjour"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="vol aggravé"
)

print(len(response.data[0].embedding))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")

print(key[:15])
print(len(key))

In [ ]:
import importlib
import rag_pipeline

importlib.reload(rag_pipeline)

result = rag_pipeline.rag_pipeline(
    "Quelle est la peine prévue pour le vol ?"
)

print(result)

In [ ]:
import importlib
import rag_pipeline

importlib.reload(rag_pipeline)

answer, docs = rag_pipeline.rag_pipeline(
    "Quelle est la peine prévue pour le vol ?"
)

print(answer)

print("\n=== SCORES ===")

for doc in docs[:3]:
    print(doc["score"])

In [ ]:
print(type(docs))

print(docs[0].keys())